# Bước 3b — Kiểm tra ngưỡng để ra quyết định làm sạch

**Mục đích.** Trước khi đặt quy tắc làm sạch ở Bước 4, notebook này kiểm tra ba nhóm giá trị bất thường: ngày sinh, thống kê xem của người dùng và mã trạng thái trong bảng danh sách. Các kết quả là bằng chứng để chọn ngưỡng, không phải để tự động xoá dữ liệu.

## 1. Ngày sinh và tuổi

So sánh phân bố tuổi trước và sau khi loại riêng năm mặc định `1930`. Điều này cho biết liệu năm đó có phải giá trị điền khuyết hay không.

In [ ]:
import pandas as pd, numpy as np

BASE = "/kaggle/input/datasets/azathoth42/myanimelist/"
users = pd.read_csv(BASE + "UserList.csv", keep_default_na=False, na_values=[""])

birth = pd.to_datetime(users["birth_date"], errors="coerce")
age = (pd.Timestamp("2018-01-01") - birth).dt.days / 365.25
age = age.dropna()
print("người dùng có ngày sinh đọc được:", len(age))
print("sinh năm 1930:", int((birth.dt.year == 1930).sum()))

a = age[(age >= 0) & (age <= 120)]
bands = pd.cut(a, bins=[0, 8, 10, 13, 18, 25, 35, 45, 55, 65, 75, 85, 120], right=False)
t = bands.value_counts().sort_index()
print(pd.DataFrame({"người dùng": t, "tỷ lệ %": (t / len(a) * 100).round(3)}))

# Không tính năm mặc định 1930
a2 = age[birth.dt.year != 1930]
a2 = a2[(a2 >= 0) & (a2 <= 120)]
t2 = pd.cut(a2, bins=[0, 8, 10, 13, 18, 25, 35, 45, 55, 65, 75, 85, 120], right=False).value_counts().sort_index()
print("\nkhông tính năm sinh 1930:")
print(pd.DataFrame({"người dùng": t2, "tỷ lệ %": (t2 / len(a2) * 100).round(3)}))

## 2. Giá trị xem bất thường

Liệt kê các hồ sơ có tổng thời gian hoặc số tập xem rất lớn. So sánh `user_days_spent_watching` với số ngày kể từ khi tham gia để xác định các giá trị không thể xảy ra về mặt thời gian.

In [ ]:
join = pd.to_datetime(users["join_date"], errors="coerce")
account_days = (pd.Timestamp("2018-05-25") - join).dt.days
users["account_days"] = account_days

cols = ["username", "user_days_spent_watching", "stats_episodes", "user_completed",
        "stats_rewatched", "join_date", "account_days"]

print("10 người dùng cao nhất theo user_days_spent_watching")
print(users.nlargest(10, "user_days_spent_watching")[cols].to_string(index=False))

print("\n10 người dùng cao nhất theo stats_episodes")
print(users.nlargest(10, "stats_episodes")[cols].to_string(index=False))

for th in [1000, 2000, 3650, 4900, 10000]:
    print(f"ngày đã xem > {th:>6,}:", int((users["user_days_spent_watching"] > th).sum()))
for th in [20_000, 50_000, 100_000, 500_000]:
    print(f"số tập đã xem > {th:>8,}:", int((users["stats_episodes"] > th).sum()))

# Không thể xảy ra: số ngày đã xem lớn hơn số ngày kể từ khi tham gia
imp = users["user_days_spent_watching"] > users["account_days"]
print("\nsố người xem nhiều ngày hơn số ngày từ lúc tham gia:", int(imp.sum()))

## 3. Mã trạng thái không thuộc miền hợp lệ

Đọc bảng 80 triệu dòng theo từng khối để tìm các mã `my_status` không thuộc tập hợp hợp lệ `{1, 2, 3, 4, 6}`. In ra số lượng, người dùng liên quan và vài dòng ví dụ để quyết định cách xử lý ở Bước 4.

In [ ]:
valid = {1, 2, 3, 4, 6}
parts, total = [], 0
usecols = ["username", "anime_id", "my_status", "my_score", "my_watched_episodes"]
for ch in pd.read_csv(BASE + "UserAnimeList.csv", usecols=usecols,
                      keep_default_na=False, na_values=[""], chunksize=5_000_000):
    total += len(ch)
    parts.append(ch[~ch["my_status"].isin(valid)])
bad = pd.concat(parts)
print("tổng số dòng:", f"{total:,}", "| trạng thái không xác định:", len(bad))
print(bad["my_status"].value_counts())
print("người dùng:", bad["username"].nunique())
print(bad["username"].value_counts().head(10))
print("dòng có điểm > 0:", int((bad["my_score"] > 0).sum()),
      "| có số tập đã xem > 0:", int((bad["my_watched_episodes"] > 0).sum()))
print(bad.head(10).to_string(index=False))

## Bước tiếp theo

Dùng các kết quả trên cùng bảng quyết định làm sạch để xây dựng và chạy Bước 4. Sau khi thay đổi một quy tắc, cần chạy lại Bước 4 để dữ liệu đầu ra và nhật ký kiểm toán khớp với quy tắc đó.